# Задание 1. Код Хаффмана для блоков из 1 и 2 символов

Берем большой текст из Project Gutenberg: **The Brothers Karamazov**.

Нужно построить код Хаффмана вручную для блоков из одного символа и для блоков из двух символов, а затем сравнить энтропию, среднюю длину кода и размер сжатого текста.

In [ ]:
import heapq
import itertools
import math
import re
from collections import Counter
from dataclasses import dataclass

import pandas as pd
import requests

## 1. Загружаем текст

In [ ]:
BOOK_URL = "https://www.gutenberg.org/ebooks/28054.txt.utf-8"

response = requests.get(BOOK_URL, timeout=30)
response.raise_for_status()

raw_text = response.text.replace("\r\n", "\n").replace("\r", "\n")


def remove_gutenberg_wrapper(text):
    start = re.search(r"\*\*\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)

    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]

    return text.strip()


text = remove_gutenberg_wrapper(raw_text)
original_size = len(text.encode("utf-8"))

print("Символов:", len(text))
print("Байт UTF-8:", original_size)
print(text[:500])

## 2. Реализация Хаффмана

In [ ]:
@dataclass
class Node:
    symbol: object = None
    left: object = None
    right: object = None

    def is_leaf(self):
        return self.symbol is not None


def split_blocks(sequence, block_size):
    return [sequence[i:i + block_size] for i in range(0, len(sequence), block_size)]


def build_huffman_tree(frequencies):
    order = itertools.count()
    heap = [(freq, next(order), Node(symbol=symbol)) for symbol, freq in frequencies.items()]
    heapq.heapify(heap)

    if len(heap) == 1:
        return heap[0][2]

    while len(heap) > 1:
        freq_1, _, node_1 = heapq.heappop(heap)
        freq_2, _, node_2 = heapq.heappop(heap)
        parent = Node(left=node_1, right=node_2)
        heapq.heappush(heap, (freq_1 + freq_2, next(order), parent))

    return heap[0][2]


def build_codes(root):
    codes = {}

    def walk(node, prefix):
        if node.is_leaf():
            codes[node.symbol] = prefix or "0"
            return

        walk(node.left, prefix + "0")
        walk(node.right, prefix + "1")

    walk(root, "")
    return codes


def encode_blocks(blocks, codes):
    return "".join(codes[block] for block in blocks)


def decode_bits(bits, root):
    result = []
    node = root

    for bit in bits:
        node = node.left if bit == "0" else node.right

        if node.is_leaf():
            result.append(node.symbol)
            node = root

    return result


def entropy(frequencies):
    total = sum(frequencies.values())
    result = 0

    for freq in frequencies.values():
        p = freq / total
        result -= p * math.log2(p)

    return result


def average_code_length(frequencies, codes):
    total = sum(frequencies.values())
    return sum((freq / total) * len(codes[symbol]) for symbol, freq in frequencies.items())

## 3. Кодируем блоками из 1 и 2 символов

In [ ]:
def huffman_experiment(text, block_size):
    blocks = split_blocks(text, block_size)
    frequencies = Counter(blocks)
    tree = build_huffman_tree(frequencies)
    codes = build_codes(tree)

    encoded_bits = encode_blocks(blocks, codes)
    decoded_text = "".join(decode_bits(encoded_bits, tree))

    assert decoded_text == text

    block_entropy = entropy(frequencies)
    block_avg_length = average_code_length(frequencies, codes)
    symbols_per_block = len(text) / len(blocks)

    return {
        "block_size": block_size,
        "alphabet_size": len(frequencies),
        "entropy_per_block": block_entropy,
        "avg_len_per_block": block_avg_length,
        "entropy_per_symbol": block_entropy / symbols_per_block,
        "avg_len_per_symbol": block_avg_length / symbols_per_block,
        "encoded_bits": len(encoded_bits),
        "compressed_bytes": math.ceil(len(encoded_bits) / 8),
        "compression_ratio": math.ceil(len(encoded_bits) / 8) / original_size,
        "codes": codes,
        "frequencies": frequencies,
    }


results = [huffman_experiment(text, 1), huffman_experiment(text, 2)]

comparison = pd.DataFrame([
    {key: value for key, value in result.items() if key not in ["codes", "frequencies"]}
    for result in results
])

comparison

## 4. Несколько самых частых блоков

In [ ]:
for result in results:
    print()
    print(f"Блоки размера {result['block_size']}")
    rows = []

    for block, freq in result["frequencies"].most_common(15):
        rows.append({
            "block": repr(block),
            "frequency": freq,
            "code": result["codes"][block],
            "code_length": len(result["codes"][block]),
        })

    display(pd.DataFrame(rows))

## Вывод

Кодирование блоками из двух символов обычно дает среднюю длину на один исходный символ ближе к энтропии, потому что алгоритм учитывает зависимости между соседними символами. При этом алфавит блоков становится больше, поэтому таблица кодов тоже растет.